# Importing libraries

In [1]:
# Basic libraries
import pandas as pd
import numpy as np
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.multioutput import MultiOutputClassifier

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss
from memory_profiler import memory_usage


# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [2]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [3]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

train = ds['train'].to_pandas()
test = ds['test'].to_pandas() 
val = ds['validation'].to_pandas()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163168 entries, 0 to 163167
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   id             163168 non-null  object             
 1   title          163168 non-null  object             
 2   abstract       163168 non-null  object             
 3   categories     163168 non-null  object             
 4   creation_date  163168 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), object(4)
memory usage: 6.2+ MB


# Dataset preprocessing

In [4]:
allowed_categories = ["cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"]

def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

train['categories'] = train['categories'].apply(clean_element)
test['categories'] = test['categories'].apply(clean_element)
val['categories'] = val['categories'].apply(clean_element)

In [5]:
train = train[train['categories'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test = test[test['categories'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val = val[val['categories'].apply(lambda cats: all(c in allowed_categories for c in cats))]

In [6]:
train.rename(columns={'categories': 'labels'}, inplace=True)
test.rename(columns={'categories': 'labels'}, inplace=True)
val.rename(columns={'categories': 'labels'}, inplace=True)

train.rename(columns={'title': 'text'}, inplace=True)
test.rename(columns={'title': 'text'}, inplace=True)
val.rename(columns={'title': 'text'}, inplace=True)

train.drop(columns=['id', 'abstract', 'creation_date'], inplace=True)
test.drop(columns=['id', 'abstract', 'creation_date'], inplace=True)
val.drop(columns=['id', 'abstract', 'creation_date'], inplace=True)

train.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)
val.reset_index(drop=True, inplace=True)

train

,text,labels
0,Upper and Lower Bounds for Large Scale Multist...,[math.OC]
1,Binary Classification: Counterbalancing Class ...,[cs.LG]
2,Smooth Optimization with Approximate Gradient,[math.OC]
3,An AI-powered Smart Routing Solution for Payme...,[cs.AI]
4,A linearly convergent method for solving high-...,[math.OC]
...,...,...
10141,Simple Question Answering with Subgraph Rankin...,"[cs.CL, cs.LG, stat.ML]"
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...","[cs.AI, cs.LG, stat.ML]"
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,"[cs.AI, cs.CL]"
10144,Near-optimal bounds for phase synchronization,[math.OC]


In [7]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train['labels'])
val_labels_binarized = mlb.transform(val['labels'])
test_labels_binarized = mlb.transform(test['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train = pd.concat([train, train_labels_df], axis=1)
val = pd.concat([val, val_labels_df], axis=1)
test = pd.concat([test, test_labels_df], axis=1)

train = train.drop(columns=['labels'])
val = val.drop(columns=['labels'])
test = test.drop(columns=['labels'])

train

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML
0,Upper and Lower Bounds for Large Scale Multist...,0,0,0,1,0
1,Binary Classification: Counterbalancing Class ...,0,0,1,0,0
2,Smooth Optimization with Approximate Gradient,0,0,0,1,0
3,An AI-powered Smart Routing Solution for Payme...,1,0,0,0,0
4,A linearly convergent method for solving high-...,0,0,0,1,0
...,...,...,...,...,...,...
10141,Simple Question Answering with Subgraph Rankin...,0,1,1,0,1
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...",1,0,1,0,1
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,1,1,0,0,0
10144,Near-optimal bounds for phase synchronization,0,0,0,1,0


In [8]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [9]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': MultiOutputClassifier(SVC()),
        'params': {
            'estimator__C': [0.1, 1, 10],
            'estimator__kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultiOutputClassifier(MultinomialNB()),
        'params': {
            'estimator__alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': MultiOutputClassifier(LogisticRegression(max_iter=1000)),
        'params': {
            'estimator__C': [0.1, 1, 10],
            'estimator__penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': MultiOutputClassifier(GradientBoostingClassifier()),
        'params': {
            'estimator__n_estimators': [100, 150, 200],
            'estimator__criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': MultiOutputClassifier(AdaBoostClassifier()),
        'params': {
            'estimator__n_estimators': [50, 100, 150],
            'estimator__learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': MultiOutputClassifier(SGDClassifier()),
        'params': {
            'estimator__alpha': [0.0001, 0.001, 0.01],
            'estimator__penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [10]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'hamming_loss', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = list(train.columns[1:])
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [11]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train[classes])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                val_classes = val.drop(columns=['text'])

                accuracy = accuracy_score(val_classes, y_pred)
                
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val_classes.to_numpy(), y_pred, average=None, zero_division=0)

                hamm_loss = hamming_loss(val_classes.to_numpy(), y_pred)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'hamming_loss': hamm_loss,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_multilabel3.csv', index=False)

Processing seed: 2
Processing vectorizer: TfidfVectorizer
Processing model: RandomForest
Training RandomForest with params {'n_estimators': 50, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 7.978967099974398
Peak memory usage during training: 755.21875 MB
Prediction time: 1.1562128000077792
Peak memory usage during prediction: 755.2890625 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'entropy'} and vectorizer TfidfVectorizer


C:\Users\Rafael\AppData\Local\Temp\ipykernel_20804\2813873047.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


Training time: 9.043904799968004
Peak memory usage during training: 754.859375 MB
Prediction time: 1.0218680999823846
Peak memory usage during prediction: 754.70703125 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'log_loss'} and vectorizer TfidfVectorizer


KeyboardInterrupt: 

# Process results

In [399]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   seed                           396 non-null    object 
 1   vectorizer                     396 non-null    object 
 2   model                          396 non-null    object 
 3   params                         396 non-null    object 
 4   accuracy                       396 non-null    float64
 5   training_time                  396 non-null    float64
 6   prediction_time                396 non-null    float64
 7   peak_memory_train              396 non-null    float64
 8   peak_memory_prediction         396 non-null    float64
 9   precision_class_toxic          396 non-null    float64
 10  recall_class_toxic             396 non-null    float64
 11  f1_class_toxic                 396 non-null    float64
 12  precision_class_severe_toxic   396 non-null    flo

In [400]:
results.head()

,seed,vectorizer,model,params,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_toxic,...,f1_class_obscene,precision_class_threat,recall_class_threat,f1_class_threat,precision_class_insult,recall_class_insult,f1_class_insult,precision_class_identity_hate,recall_class_identity_hate,f1_class_identity_hate
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.534206,1.149115,1.334285,832.828125,835.640625,0.813614,...,0.530559,0.0,0.0,0.0,0.758904,0.177678,0.287942,1.0,0.003817,0.007605
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.539753,0.893782,1.332878,835.644531,834.722656,0.802174,...,0.491921,0.0,0.0,0.0,0.788413,0.200770,0.320041,1.0,0.003817,0.007605
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.539753,1.154756,1.348080,833.531250,832.558594,0.802174,...,0.491921,0.0,0.0,0.0,0.788413,0.200770,0.320041,1.0,0.003817,0.007605
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.545917,1.329363,0.729114,826.187500,832.937500,0.892568,...,0.524978,0.0,0.0,0.0,0.769231,0.166774,0.274117,0.0,0.000000,0.000000
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.544992,0.704777,0.733598,832.941406,825.183594,0.798498,...,0.508913,0.0,0.0,0.0,0.804598,0.179602,0.293655,1.0,0.003817,0.007605


In [401]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_toxic,...,precision_class_threat,recall_class_threat,f1_class_threat,precision_class_insult,recall_class_insult,f1_class_insult,precision_class_identity_hate,recall_class_identity_hate,f1_class_identity_hate,f1_avg
0,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.534977,1.518558,0.967381,829.490885,829.442708,0.964796,...,0.033086,0.015152,0.020733,0.623077,0.051956,0.095915,0.227273,0.019084,0.035211,0.221203
1,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.361839,1.546520,1.039813,829.442708,829.446615,0.604374,...,0.010417,0.003788,0.005556,0.673522,0.336113,0.448435,0.048204,0.194656,0.077273,0.332759
2,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.343760,1.623639,1.006274,829.444010,829.464844,0.625012,...,0.017544,0.003788,0.006231,0.565009,0.481291,0.519790,0.162043,0.162850,0.144231,0.366275
3,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 150, 'estimator__l...",3.333333,0.540010,2.108233,1.156371,829.464844,829.510417,0.962065,...,0.030025,0.011364,0.016485,0.714286,0.093008,0.164586,0.250000,0.022901,0.041958,0.235333
4,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 150, 'estimator__l...",3.333333,0.455316,2.075261,1.160997,829.506510,829.764323,0.691897,...,0.020833,0.007576,0.011111,0.653311,0.418217,0.509965,0.099417,0.167939,0.124891,0.343738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'estimator__C': 1, 'estimator__kernel': 'rbf'}",3.333333,0.538367,1.026851,1.557175,823.078125,829.653646,0.908851,...,0.000000,0.000000,0.000000,0.868132,0.050674,0.095758,0.000000,0.000000,0.000000,0.165338
128,SVC,TfidfVectorizer,"{'estimator__C': 1, 'estimator__kernel': 'sigm...",3.333333,0.556394,0.998959,1.107137,824.958333,829.528646,0.814186,...,0.000000,0.000000,0.000000,0.785064,0.276459,0.408918,0.333333,0.007634,0.014925,0.282629
129,SVC,TfidfVectorizer,"{'estimator__C': 10, 'estimator__kernel': 'lin...",3.333333,0.531587,1.006885,1.150323,823.105469,829.643229,0.753951,...,0.000000,0.000000,0.000000,0.670412,0.459269,0.545108,0.361702,0.129771,0.191011,0.388972
130,SVC,TfidfVectorizer,"{'estimator__C': 10, 'estimator__kernel': 'rbf'}",3.333333,0.555624,1.024329,1.563466,823.123698,829.597656,0.823787,...,0.000000,0.000000,0.000000,0.843666,0.200770,0.324352,0.555556,0.019084,0.036900,0.268598


In [406]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train[classes])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test[classes], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test[classes], y_pred, average=None, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: MultinomialNB
Best model params: {'estimator__alpha': 0.1}
Best vectorizer: CountVectorizer
Best accuracy: 0.7611522710931883

Class toxic
Precision: 0.32996920971604515
Recall: 0.6334975369458128
F1: 0.4339219435384096
Support: 6090

Class severe_toxic
Precision: 0.1549876339653751
Recall: 0.5122615803814714
F1: 0.2379746835443038
Support: 367

Class obscene
Precision: 0.36729435084241824
Recall: 0.5020319696559198
F1: 0.4242216117216117
Support: 3691

Class threat
Precision: 0.023391812865497075
Recall: 0.037914691943127965
F1: 0.028933092224231464
Support: 211

Class insult
Precision: 0.28019158398905236
Recall: 0.47796906915669685
F1: 0.3532837269492074
Support: 3427

Class identity_hate
Precision: 0.0982749607945635
Recall: 0.2640449438202247
F1: 0.14323809523809525
Support: 712



In [407]:
with open('models/best_model_sklearn_multilabel3.pkl', 'wb') as f:
    pickle.dump(pipeline, f)